<a href="https://colab.research.google.com/github/Benzsoft/ai-qos-classification-5g/blob/main/colab-foundation/notebooks/00_colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 — Colab research setup
Run cells in order. Select a GPU runtime for the TensorFlow check. This notebook preserves installed packages and records their versions. It does not train models or download large datasets. Google Drive access is requested to preserve outputs. The original six model families are retained.


In [1]:
from pathlib import Path
import subprocess, sys, json, datetime, platform, importlib.metadata
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/5G_QoS_Research')
for name in ['environment', 'audits', 'data/manifests', 'configs', 'results', 'models', 'figures']:
    (PROJECT / name).mkdir(parents=True, exist_ok=True)
print('Persistent project:', PROJECT)


Mounted at /content/drive
Persistent project: /content/drive/MyDrive/5G_QoS_Research


In [2]:
REPO = Path('/content/ai-qos-research')
URL = 'https://github.com/Benzsoft/ai-qos-classification-5g.git'
REF = 'research/colab-foundation'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', REF, '--single-branch', URL, str(REPO)], check=True)
else:
    origin = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
    assert origin == URL, f'Unexpected repository: {origin}'
    print('Reusing checkout without overwriting local work.')
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Exact research commit:', COMMIT)


Exact research commit: 89fc45241e08dbae4fa96594a02888d9fbf7f287


In [3]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as error:
        print('GPU already initialized:', error)
if gpus:
    with tf.device('/GPU:0'):
        result = tf.matmul(tf.ones((128, 128)), tf.ones((128, 128)))
    assert 'GPU' in result.device, result.device
    print('GPU verified:', result.device)
else:
    print('CPU mode: dataset auditing and classical models can still run.')


TensorFlow: 2.20.0
GPU verified: /job:localhost/replica:0/task:0/device:GPU:0


In [4]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
record = {'timestamp_utc': stamp, 'python': sys.version, 'platform': platform.platform(), 'git_commit': COMMIT}
record['packages'] = {p: importlib.metadata.version(p) for p in ['tensorflow','scikit-learn','pandas','numpy']}
try:
    record['gpu'] = subprocess.check_output(['nvidia-smi'], text=True)
except FileNotFoundError:
    record['gpu'] = None
record['tensorflow_build'] = tf.sysconfig.get_build_info()
(PROJECT / 'environment' / f'{stamp}.json').write_text(json.dumps(record, indent=2, default=str))
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(PROJECT / 'environment' / f'{stamp}_packages.txt').write_text(freeze)
print(json.dumps(record['packages'], indent=2))
print('Setup complete. Open notebook 01 next. Package snapshot is a record, not a portable lockfile.')


{
  "tensorflow": "2.20.0",
  "scikit-learn": "1.6.1",
  "pandas": "2.2.3",
  "numpy": "2.1.3"
}
Setup complete. Open notebook 01 next. Package snapshot is a record, not a portable lockfile.
